# Step 2 of RAG: Storing Chunks in ChromaDB**RAG** (Retrieval-Augmented Generation) pipeline recap:1. Document Loading & Chunking (Step 1)2. **Storing chunks in a Vector Database** ← *this notebook*3. Retrieval (searching the vector database for relevant chunks)4. Passing retrieved chunks to an LLM to generate the final answerThis notebook is self-contained — it recreates a sample document, chunks it (recap of Step 1),and then focuses fully on **storing those chunks in ChromaDB** the right way: with embeddings,useful metadata, and stable IDs so we can retrieve, update, or trace every chunk back to its source.### What you will learn1. Recap: create and chunk a document2. Why metadata (source, chunk index, etc.) matters when storing chunks3. Setting up a ChromaDB collection with a Hugging Face embedding function4. Storing chunks with meaningful IDs and metadata5. Verifying what got stored6. Querying stored chunks (a preview of Step 3: Retrieval)7. Adding a second document to show multi-document storage

### Setup — Install Required Libraries

In [ ]:
!pip install -q langchain-text-splitters chromadb sentence-transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 969.6 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 45.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 59.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 44.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.1/140.1 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.3/206.3 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.7/95.7 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 2.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently ta

## 1. Recap — Create and Chunk a Document (Step 1)We regenerate a small sample document and split it into chunks using`RecursiveCharacterTextSplitter`, exactly as in the Step 1 notebook.

In [ ]:
import os

os.makedirs("docs", exist_ok=True)

article_text = """Artificial Intelligence: An Overview

Artificial Intelligence, or AI, is the field of computer science focused on building systems that can perform tasks which normally require human intelligence. These tasks include understanding language, recognizing images, making decisions, and learning from experience.

Machine Learning is a subset of AI where systems learn patterns from data instead of following hardcoded rules. Instead of programming every decision by hand, we feed the system many examples, and it learns the underlying pattern on its own. This approach powers everything from spam filters to recommendation systems.

Deep Learning takes Machine Learning further by using neural networks with many layers. These layered networks can automatically discover complex patterns in large amounts of data, such as recognizing faces in photos or understanding the meaning of a sentence.

Natural Language Processing, or NLP, is the branch of AI that focuses specifically on human language. NLP powers chatbots, translation tools, and search engines. A key building block of modern NLP is the embedding, which converts words or sentences into numeric vectors that capture meaning.

Retrieval-Augmented Generation, or RAG, is a technique that combines a search system with a language model. Instead of relying only on what a language model memorized during training, RAG first retrieves relevant, up-to-date information from a document collection, then passes that information to the model so it can generate a more accurate, grounded answer.

Vector Databases store embeddings and allow fast similarity search across millions of documents. They are a core infrastructure piece behind modern RAG systems, chatbots, and recommendation engines used by companies around the world today."""

with open("docs/ai_overview.txt", "w", encoding="utf-8") as f:
    f.write(article_text)

print("Document created: docs/ai_overview.txt")


Document created: docs/ai_overview.txt


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50,
    separators=["\n\n", "\n", ". ", " ", ""]
)

chunks = splitter.split_text(article_text)

print("Number of chunks created:", len(chunks))
for i, chunk in enumerate(chunks):
    print(f"\n--- Chunk {i} ({len(chunk)} chars) ---")
    print(chunk)


Number of chunks created: 9

--- Chunk 0 (36 chars) ---
Artificial Intelligence: An Overview

--- Chunk 1 (270 chars) ---
Artificial Intelligence, or AI, is the field of computer science focused on building systems that can perform tasks which normally require human intelligence. These tasks include understanding language, recognizing images, making decisions, and learning from experience.

--- Chunk 2 (240 chars) ---
Machine Learning is a subset of AI where systems learn patterns from data instead of following hardcoded rules. Instead of programming every decision by hand, we feed the system many examples, and it learns the underlying pattern on its own

--- Chunk 3 (78 chars) ---
. This approach powers everything from spam filters to recommendation systems.

--- Chunk 4 (260 chars) ---
Deep Learning takes Machine Learning further by using neural networks with many layers. These layered networks can automatically discover complex patterns in large amounts of data, such as recognizing 

## 2. Why Metadata Matters When Storing ChunksWhen we store a chunk in a vector database, we should **never** store just the raw text. We shouldalso attach metadata so that later, when a chunk is retrieved, we know:| Metadata field | Why it matters ||---|---|| `source` | Which original file/document this chunk came from || `chunk_index` | The chunk's position/order within that document || `char_count` | Useful for debugging or filtering overly short/long chunks |This makes results traceable — a RAG answer can cite "this came from `ai_overview.txt`, chunk 3".

In [ ]:
chunk_metadatas = [
    {
        "source": "ai_overview.txt",
        "chunk_index": i,
        "char_count": len(chunk)
    }
    for i, chunk in enumerate(chunks)
]

chunk_ids = [f"ai_overview_chunk_{i}" for i in range(len(chunks))]

print("Example metadata for chunk 0:", chunk_metadatas[0])
print("Example id for chunk 0:", chunk_ids[0])


Example metadata for chunk 0: {'source': 'ai_overview.txt', 'chunk_index': 0, 'char_count': 36}
Example id for chunk 0: ai_overview_chunk_0


## 3. Set Up ChromaDB with a Hugging Face Embedding FunctionWe use the free `all-MiniLM-L6-v2` model so ChromaDB automatically converts every chunk's text intoan embedding vector as we add it.

In [ ]:
import chromadb
from chromadb.utils import embedding_functions

client = chromadb.Client()

hf_embedder = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="all-MiniLM-L6-v2"
)

collection = client.get_or_create_collection(
    name="rag_chunks",
    embedding_function=hf_embedder
)

print("Collection created:", collection.name)


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Collection created: rag_chunks


## 4. Store the Chunks in ChromaDBWe pass `documents` (the chunk text), `metadatas`, and `ids` all together. ChromaDB embeds eachchunk automatically using the embedding function we attached above.

In [ ]:
collection.add(
    documents=chunks,
    metadatas=chunk_metadatas,
    ids=chunk_ids
)

print("Total chunks stored in ChromaDB:", collection.count())


Total chunks stored in ChromaDB: 9


## 5. Verify What Got StoredAlways double-check after a storage step — inspect a few chunks along with their metadata.

In [ ]:
stored = collection.get(
    ids=chunk_ids,
    include=["documents", "metadatas"]
)

for cid, doc, meta in zip(
    stored["ids"],
    stored["documents"],
    stored["metadatas"]
):
    print(f"ID: {cid}")
    print(f"Metadata: {meta}")
    print(f"Text: {doc[:100]}...")
    print("-" * 60)


ID: ai_overview_chunk_0
Metadata: {'char_count': 36, 'source': 'ai_overview.txt', 'chunk_index': 0}
Text: Artificial Intelligence: An Overview...
------------------------------------------------------------
ID: ai_overview_chunk_1
Metadata: {'chunk_index': 1, 'source': 'ai_overview.txt', 'char_count': 270}
Text: Artificial Intelligence, or AI, is the field of computer science focused on building systems that ca...
------------------------------------------------------------
ID: ai_overview_chunk_2
Metadata: {'char_count': 240, 'chunk_index': 2, 'source': 'ai_overview.txt'}
Text: Machine Learning is a subset of AI where systems learn patterns from data instead of following hardc...
------------------------------------------------------------
ID: ai_overview_chunk_3
Metadata: {'chunk_index': 3, 'char_count': 78, 'source': 'ai_overview.txt'}
Text: . This approach powers everything from spam filters to recommendation systems....
------------------------------------------------------------


## 6. Query the Stored Chunks (Preview of Step 3: Retrieval)Now that our chunks live in ChromaDB, we can search them by meaning. This is the foundation of thenext RAG step — retrieving the most relevant chunks for a user's question.

In [ ]:
results = collection.query(
    query_texts=["How does RAG combine search with language models?"],
    n_results=2,
    include=["documents", "metadatas", "distances"]
)

for doc, meta, distance in zip(
    results["documents"][0],
    results["metadatas"][0],
    results["distances"][0]
):
    print(
        f"Distance: {distance:.4f} | "
        f"Source: {meta['source']} | "
        f"Chunk #{meta['chunk_index']}"
    )
    print(doc)
    print("-" * 60)


Distance: 0.2155 | Source: ai_overview.txt | Chunk #6
Retrieval-Augmented Generation, or RAG, is a technique that combines a search system with a language model
------------------------------------------------------------
Distance: 0.3735 | Source: ai_overview.txt | Chunk #7
. Instead of relying only on what a language model memorized during training, RAG first retrieves relevant, up-to-date information from a document collection, then passes that information to the model so it can generate a more accurate, grounded answer.
------------------------------------------------------------


**Discussion Point:** The `distance` score tells us how relevant each chunk is (lower = morerelevant). In a full RAG system, these top chunks would now be inserted into a prompt and sent to anLLM to generate a grounded final answer — that's Step 3 and Step 4.

## 7. Adding a Second Document (Multi-Document Storage)Real RAG systems store chunks from many documents in the same collection. The `source` metadata fieldis what lets us tell them apart later.

In [ ]:
second_doc_text = """Embeddings and Vector Search in Practice

Modern search engines increasingly rely on embeddings rather than simple keyword matching. An embedding model converts a search query into the same vector space as the stored documents, allowing the system to find results that match the meaning of the query, not just its exact words. This is especially powerful for customer support systems, where a user might phrase a question very differently from how it appears in a help article, yet still expect to find the right answer."""

with open("docs/embeddings_in_practice.txt", "w", encoding="utf-8") as f:
    f.write(second_doc_text)

second_doc_chunks = splitter.split_text(second_doc_text)

second_doc_metadatas = [
    {
        "source": "embeddings_in_practice.txt",
        "chunk_index": i,
        "char_count": len(chunk)
    }
    for i, chunk in enumerate(second_doc_chunks)
]

second_doc_ids = [
    f"embeddings_in_practice_chunk_{i}"
    for i in range(len(second_doc_chunks))
]

collection.add(
    documents=second_doc_chunks,
    metadatas=second_doc_metadatas,
    ids=second_doc_ids
)

print("Chunks from second document added:", len(second_doc_chunks))
print("Total chunks now stored in ChromaDB:", collection.count())


Chunks from second document added: 3
Total chunks now stored in ChromaDB: 12


In [ ]:
# Query across BOTH documents at once
results = collection.query(
    query_texts=["how do search engines use embeddings"],
    n_results=3,
    include=["documents", "metadatas", "distances"]
)

for doc, meta, distance in zip(
    results["documents"][0],
    results["metadatas"][0],
    results["distances"][0]
):
    print(
        f"Distance: {distance:.4f} | "
        f"Source: {meta['source']} | "
        f"Chunk #{meta['chunk_index']}"
    )
    print(doc)
    print("-" * 60)


Distance: 0.1885 | Source: embeddings_in_practice.txt | Chunk #1
Modern search engines increasingly rely on embeddings rather than simple keyword matching. An embedding model converts a search query into the same vector space as the stored documents, allowing the system to find results that match the meaning of the query, not just its exact words
------------------------------------------------------------
Distance: 0.2940 | Source: embeddings_in_practice.txt | Chunk #0
Embeddings and Vector Search in Practice
------------------------------------------------------------
Distance: 0.4229 | Source: ai_overview.txt | Chunk #8
Vector Databases store embeddings and allow fast similarity search across millions of documents. They are a core infrastructure piece behind modern RAG systems, chatbots, and recommendation engines used by companies around the world today.
------------------------------------------------------------


**Discussion Point:** Notice the top result correctly comes from `embeddings_in_practice.txt` eventhough our collection also contains chunks from `ai_overview.txt`. This is exactly how a RAG systemsearches across an entire knowledge base to find the most relevant pieces, regardless of source.

## 8. Filtering Retrieval to a Single Source (Optional but Useful)Sometimes you only want to search within one document — combine semantic search with a metadatafilter using `where`.

In [ ]:
results = collection.query(
    query_texts=["what is deep learning"],
    n_results=2,
    where={"source": "ai_overview.txt"},
    include=["documents", "metadatas", "distances"]
)

for doc, meta, distance in zip(
    results["documents"][0],
    results["metadatas"][0],
    results["distances"][0]
):
    print(
        f"Distance: {distance:.4f} | "
        f"Source: {meta['source']} | "
        f"Chunk #{meta['chunk_index']}"
    )
    print(doc)
    print("-" * 60)


Distance: 0.3109 | Source: ai_overview.txt | Chunk #4
Deep Learning takes Machine Learning further by using neural networks with many layers. These layered networks can automatically discover complex patterns in large amounts of data, such as recognizing faces in photos or understanding the meaning of a sentence.
------------------------------------------------------------
Distance: 0.5647 | Source: ai_overview.txt | Chunk #2
Machine Learning is a subset of AI where systems learn patterns from data instead of following hardcoded rules. Instead of programming every decision by hand, we feed the system many examples, and it learns the underlying pattern on its own
------------------------------------------------------------


## 9. Practice Exercises — Solutions

The following cells complete the three practice exercises from this notebook.

## Key Takeaways (Recap for Students)| Concept | What We Did ||---|---|| Chunk storage | Stored each chunk as a `document` in ChromaDB, not the whole file at once || Embedding function | Attached `all-MiniLM-L6-v2` so ChromaDB embeds text automatically on `add()` || Metadata | Stored `source`, `chunk_index`, `char_count` with every chunk for traceability || IDs | Used predictable ids like `ai_overview_chunk_0` so chunks can be updated/deleted later || Multi-document storage | Added a second document into the same collection || Filtered retrieval | Used `where={"source": ...}` to search within a single document only |This collection of stored chunks is now ready for **Step 3: Retrieval**, where we take a user'squestion, search this same collection, and pass the top chunks to an LLM to generate the final answer.### Practice Exercises1. Add a third document of your own and confirm `collection.count()` reflects all chunks from all three.2. Re-chunk `ai_overview.txt` with `chunk_size=150` and re-store it under new ids (e.g. `..._v2_chunk_0`)   — compare retrieval results between the two chunk sizes.3. Write a helper function `add_document(collection, splitter, text, source_name)` that handles chunking,   metadata creation, and `collection.add()` in one reusable call.

In [ ]:
# Exercise 1 — Add a third document and verify the total count

third_doc_text = """Generative AI and Modern Applications

Generative AI models can create new text, images, audio, and code from learned patterns.
They are commonly used for assistants, content generation, summarization, and software development."""

third_doc_chunks = splitter.split_text(third_doc_text)

third_doc_metadatas = [
    {
        "source": "generative_ai.txt",
        "chunk_index": i,
        "char_count": len(chunk)
    }
    for i, chunk in enumerate(third_doc_chunks)
]

third_doc_ids = [
    f"generative_ai_chunk_{i}"
    for i in range(len(third_doc_chunks))
]

collection.add(
    documents=third_doc_chunks,
    metadatas=third_doc_metadatas,
    ids=third_doc_ids
)

print("Third-document chunks added:", len(third_doc_chunks))
print("Total chunks across all three documents:", collection.count())


Third-document chunks added: 1
Total chunks across all three documents: 13


In [ ]:
# Exercise 2 — Re-chunk the first document with chunk_size=150
# and store the new version under different IDs.

small_splitter = RecursiveCharacterTextSplitter(
    chunk_size=150,
    chunk_overlap=30,
    separators=["\n\n", "\n", ". ", " ", ""]
)

ai_v2_chunks = small_splitter.split_text(article_text)

ai_v2_metadatas = [
    {
        "source": "ai_overview.txt",
        "chunk_index": i,
        "char_count": len(chunk),
        "version": "v2"
    }
    for i, chunk in enumerate(ai_v2_chunks)
]

ai_v2_ids = [
    f"ai_overview_v2_chunk_{i}"
    for i in range(len(ai_v2_chunks))
]

collection.add(
    documents=ai_v2_chunks,
    metadatas=ai_v2_metadatas,
    ids=ai_v2_ids
)

print("Original chunk count:", len(chunks))
print("New chunk count (150 chars):", len(ai_v2_chunks))
print("Total chunks after storing v2:", collection.count())

comparison_results = collection.query(
    query_texts=["what is deep learning"],
    n_results=3,
    where={"source": "ai_overview.txt"},
    include=["documents", "metadatas", "distances"]
)

print("\nTop results after storing both versions:")
for doc, meta, distance in zip(
    comparison_results["documents"][0],
    comparison_results["metadatas"][0],
    comparison_results["distances"][0]
):
    print(
        f"Distance: {distance:.4f} | "
        f"Version: {meta.get('version', 'original')} | "
        f"Chunk #{meta['chunk_index']}"
    )
    print(doc)
    print("-" * 60)


Original chunk count: 9
New chunk count (150 chars): 18
Total chunks after storing v2: 31

Top results after storing both versions:
Distance: 0.3109 | Version: original | Chunk #4
Deep Learning takes Machine Learning further by using neural networks with many layers. These layered networks can automatically discover complex patterns in large amounts of data, such as recognizing faces in photos or understanding the meaning of a sentence.
------------------------------------------------------------
Distance: 0.3714 | Version: v2 | Chunk #7
Deep Learning takes Machine Learning further by using neural networks with many layers
------------------------------------------------------------
Distance: 0.5647 | Version: original | Chunk #2
Machine Learning is a subset of AI where systems learn patterns from data instead of following hardcoded rules. Instead of programming every decision by hand, we feed the system many examples, and it learns the underlying pattern on its own
-------------------

In [ ]:
# Exercise 3 — Reusable helper for chunking and storing a document

def add_document(collection, splitter, text, source_name):
    """Split a document, create metadata/IDs, and store its chunks in ChromaDB."""
    document_chunks = splitter.split_text(text)

    document_metadatas = [
        {
            "source": source_name,
            "chunk_index": i,
            "char_count": len(chunk)
        }
        for i, chunk in enumerate(document_chunks)
    ]

    # Make IDs safe and predictable while preserving the source name.
    source_stem = os.path.splitext(os.path.basename(source_name))[0]
    document_ids = [
        f"{source_stem}_chunk_{i}"
        for i in range(len(document_chunks))
    ]

    collection.add(
        documents=document_chunks,
        metadatas=document_metadatas,
        ids=document_ids
    )

    return document_chunks, document_metadatas, document_ids


# Example usage with a new document.
helper_text = """Vector databases are designed to store vector representations
and efficiently retrieve records that are semantically similar to a query."""

helper_chunks, helper_metadatas, helper_ids = add_document(
    collection=collection,
    splitter=splitter,
    text=helper_text,
    source_name="vector_databases.txt"
)

print("Helper added:", len(helper_chunks), "chunks")
print("First generated ID:", helper_ids[0])
print("Total chunks in collection:", collection.count())


Helper added: 1 chunks
First generated ID: vector_databases_chunk_0
Total chunks in collection: 32
